# FastFlow + WideResNet50-2 Training

This notebook runs a FastFlow-style anomaly detector on the shared `64x64` WM-811K split using a frozen ImageNet-pretrained `WideResNet50-2` backbone.

Workflow:

- load the shared processed metadata and wafer arrays
- train only on normal wafers from the training split
- early-stop on validation-normal negative log-likelihood
- derive the deployed anomaly threshold from validation-normal scores at the configured quantile
- evaluate the test split with the same threshold rule used elsewhere in the repo
- save both the default FastFlow score and a small score-reduction sweep for comparison

In [ ]:
from __future__ import annotations
import copy
import json
import math
import random
import sys
import time
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchvision.models import Wide_ResNet50_2_Weights, wide_resnet50_2
from torchvision.models.feature_extraction import create_feature_extractor
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation import summarize_threshold_metrics, sweep_threshold_metrics
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
REPO_ROOT

In [ ]:
RUN_TRAINING = False
print(f'RUN_TRAINING = {RUN_TRAINING}')


In [ ]:
try:
    CONFIG = {'run': {'output_dir': 'artifacts/x64/fastflow_wideresnet50', 'seed': 42, 'run_training': True, 'run_score_sweep': True}, 'data': {'metadata_csv': 'data/processed/x64/wm811k/metadata_50k_5pct.csv', 'image_size': 64, 'batch_size': 16, 'num_workers': 4}, 'training': {'device': 'auto', 'epochs': 20, 'learning_rate': 0.0001, 'weight_decay': 1e-05, 'early_stopping_patience': 4, 'grad_clip_norm': 1.0, 'amp': True}, 'model': {'pretrained': True, 'input_size': 224, 'layers': ['layer2', 'layer3'], 'flow_steps': 6, 'hidden_channels': 256, 'scale_clamp': 1.9}, 'scoring': {'threshold_quantile': 0.95, 'default_reduction': 'topk_mean', 'default_topk_ratio': 0.1, 'sweep_variants': [{'name': 'mean', 'reduction': 'mean', 'topk_ratio': 0.1}, {'name': 'max', 'reduction': 'max', 'topk_ratio': 0.1}, {'name': 'topk_r005', 'reduction': 'topk_mean', 'topk_ratio': 0.05}, {'name': 'topk_r010', 'reduction': 'topk_mean', 'topk_ratio': 0.1}, {'name': 'topk_r015', 'reduction': 'topk_mean', 'topk_ratio': 0.15}, {'name': 'topk_r020', 'reduction': 'topk_mean', 'topk_ratio': 0.2}]}}
    CONFIG
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "config = {'run': {'output_dir': 'artifacts/x64/fastflow_wideresnet50', 'seed': 42, 'run_training': true, 'run_score_sweep': true}, 'data': {'metadata_csv': 'data/processed/x64/wm811k/metadata_50k_5pct.csv', 'image_size': 64, 'batch_size': 16, 'num_workers': 4}, 'training': {'device': 'auto', 'epochs': 20, 'learning_rate': 0.0001, 'weight_decay': 1e-05, 'early_stopping_patience': 4, 'grad_clip_norm': 1.0, 'amp': true}, 'model': {'pretrained': true, 'input_size': 224, 'layers': ['layer2', 'layer3'], 'flow_steps': 6, 'hidden_channels': 256, 'scale_clamp': 1.9}, 'scoring': {'threshold_quantile': 0.95, 'default_reduction': 'topk_mean', 'default_topk_ratio': 0.1, 'sweep_variants': [{'name': 'mean', 'reduction': 'mean', 'topk_ratio': 0.1}, {'name': 'max', 'reduction': 'max', 'topk_ratio': 0.1}, {'name': 'topk_r005', 'reduction': 'topk_mean', 'topk_ratio': 0.05}, {'name': 'topk_r010', 'reduction': 'topk_mean', 'topk_ratio': 0.1}, {'name': 'topk_r015', 'reduction': 'topk_mean', 'topk_ratio': 0.15}, {'name': 'topk_r020', 'reduction': 'topk_mean', 'topk_ratio': 0.2}]}}\nconfig"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
LOG_2PI = math.log(2.0 * math.pi)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(device_name)

class ImageNetWaferPreprocessor(nn.Module):

    def __init__(self, input_size: int) -> None:
        super().__init__()
        self.input_size = input_size
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(self.input_size, self.input_size), mode='bilinear', align_corners=False)
        return (x - self.mean) / self.std

class FrozenWideResNet50Backbone(nn.Module):

    def __init__(self, layers: list[str], input_size: int, pretrained: bool=True) -> None:
        super().__init__()
        weights = Wide_ResNet50_2_Weights.DEFAULT if pretrained else None
        backbone = wide_resnet50_2(weights=weights)
        self.preprocess = ImageNetWaferPreprocessor(input_size)
        self.layers = list(layers)
        self.extractor = create_feature_extractor(backbone, return_nodes={layer: layer for layer in self.layers})
        self.extractor.eval()
        for parameter in self.extractor.parameters():
            parameter.requires_grad_(False)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        x = self.preprocess(x)
        with torch.no_grad():
            return self.extractor(x)

class Orthogonal1x1Conv(nn.Module):

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.weight_raw = nn.Parameter(torch.randn(channels, channels))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        q, _ = torch.linalg.qr(self.weight_raw)
        weight = q.view(q.shape[0], q.shape[1], 1, 1)
        z = F.conv2d(x, weight)
        logdet = torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)
        return (z, logdet)

class AffineCoupling(nn.Module):

    def __init__(self, channels: int, hidden_channels: int, scale_clamp: float) -> None:
        super().__init__()
        if channels % 2 != 0:
            raise ValueError(f'FastFlow coupling expects an even number of channels, got {channels}')
        self.scale_clamp = scale_clamp
        self.half_channels = channels // 2
        self.net = nn.Sequential(nn.Conv2d(self.half_channels, hidden_channels, kernel_size=3, padding=1), nn.GELU(), nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1), nn.GELU(), nn.Conv2d(hidden_channels, channels, kernel_size=3, padding=1))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x_a, x_b = torch.chunk(x, 2, dim=1)
        shift, log_scale = torch.chunk(self.net(x_a), 2, dim=1)
        log_scale = torch.tanh(log_scale) * self.scale_clamp
        y_b = x_b * torch.exp(log_scale) + shift
        logdet = log_scale.flatten(1).sum(dim=1)
        return (torch.cat([x_a, y_b], dim=1), logdet)

class FlowStep(nn.Module):

    def __init__(self, channels: int, hidden_channels: int, scale_clamp: float) -> None:
        super().__init__()
        self.mix = Orthogonal1x1Conv(channels)
        self.coupling = AffineCoupling(channels, hidden_channels=hidden_channels, scale_clamp=scale_clamp)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z, logdet_mix = self.mix(x)
        z, logdet_coupling = self.coupling(z)
        return (z, logdet_mix + logdet_coupling)

class FastFlowStage(nn.Module):

    def __init__(self, channels: int, flow_steps: int, hidden_channels: int, scale_clamp: float) -> None:
        super().__init__()
        self.steps = nn.ModuleList([FlowStep(channels, hidden_channels=hidden_channels, scale_clamp=scale_clamp) for _ in range(flow_steps)])

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        total_logdet = torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)
        z = x
        for step in self.steps:
            z, logdet = step(z)
            total_logdet = total_logdet + logdet
        return (z, total_logdet)

class FastFlowModel(nn.Module):

    def __init__(self, backbone: FrozenWideResNet50Backbone, image_size: int, flow_steps: int, hidden_channels: int, scale_clamp: float) -> None:
        super().__init__()
        self.backbone = backbone
        with torch.no_grad():
            dummy = torch.zeros(1, 1, image_size, image_size)
            feature_shapes = self.backbone(dummy)
        self.stages = nn.ModuleDict({name: FastFlowStage(channels=feature.shape[1], flow_steps=flow_steps, hidden_channels=hidden_channels, scale_clamp=scale_clamp) for name, feature in feature_shapes.items()})

    def forward(self, x: torch.Tensor) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
        features = self.backbone(x)
        zs: list[torch.Tensor] = []
        logdets: list[torch.Tensor] = []
        for name, feature in features.items():
            z, logdet = self.stages[name](feature)
            zs.append(z)
            logdets.append(logdet)
        return (zs, logdets)

    def anomaly_map(self, x: torch.Tensor, output_size: int) -> torch.Tensor:
        zs, _ = self.forward(x)
        maps = []
        for z in zs:
            stage_map = 0.5 * z.pow(2).mean(dim=1, keepdim=True)
            stage_map = F.interpolate(stage_map, size=(output_size, output_size), mode='bilinear', align_corners=False)
            maps.append(stage_map)
        return torch.stack(maps, dim=0).mean(dim=0)

def compute_fastflow_loss(zs: list[torch.Tensor], logdets: list[torch.Tensor]) -> torch.Tensor:
    losses = []
    for z, logdet in zip(zs, logdets):
        elements_per_sample = z[0].numel()
        nll = 0.5 * (z.pow(2) + LOG_2PI).flatten(1).sum(dim=1) - logdet
        losses.append((nll / elements_per_sample).mean())
    return torch.stack(losses).mean()

def reduce_anomaly_map(anomaly_map: torch.Tensor, reduction: str, topk_ratio: float) -> torch.Tensor:
    flat = anomaly_map.flatten(1)
    if reduction == 'mean':
        return flat.mean(dim=1)
    if reduction == 'max':
        return flat.max(dim=1).values
    if reduction == 'topk_mean':
        k = max(1, int(math.ceil(flat.shape[1] * topk_ratio)))
        return flat.topk(k=k, dim=1).values.mean(dim=1)
    raise ValueError(f'Unsupported reduction: {reduction}')

In [ ]:
try:
    if RUN_TRAINING:

        def make_dataloaders(config: dict[str, Any], device: torch.device) -> dict[str, DataLoader]:
            metadata_csv = str(REPO_ROOT / config['data']['metadata_csv'])
            image_size = config['data']['image_size']
            batch_size = config['data']['batch_size']
            num_workers = config['data']['num_workers']
            pin_memory = device.type == 'cuda'
            train_dataset = WaferMapDataset(metadata_csv=metadata_csv, split='train', image_size=image_size)
            val_dataset = WaferMapDataset(metadata_csv=metadata_csv, split='val', image_size=image_size)
            test_dataset = WaferMapDataset(metadata_csv=metadata_csv, split='test', image_size=image_size)
            return {'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory), 'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory), 'test': DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)}

        def run_epoch(model: FastFlowModel, loader: DataLoader, device: torch.device, optimizer: torch.optim.Optimizer | None, scaler: torch.cuda.amp.GradScaler, amp_enabled: bool, grad_clip_norm: float | None) -> float:
            is_training = optimizer is not None
            model.train(is_training)
            running_loss = 0.0
            total_items = 0
            for inputs, _ in tqdm(loader, desc='train' if is_training else 'eval', leave=False):
                inputs = inputs.to(device, non_blocking=True)
                with torch.set_grad_enabled(is_training):
                    with torch.cuda.amp.autocast(enabled=amp_enabled):
                        zs, logdets = model(inputs)
                        loss = compute_fastflow_loss(zs, logdets)
                    if is_training:
                        optimizer.zero_grad(set_to_none=True)
                        scaler.scale(loss).backward()
                        if grad_clip_norm is not None:
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                        scaler.step(optimizer)
                        scaler.update()
                batch_size = inputs.shape[0]
                running_loss += float(loss.detach().cpu()) * batch_size
                total_items += batch_size
            return running_loss / max(1, total_items)

        def collect_anomaly_maps(model: FastFlowModel, loader: DataLoader, device: torch.device, image_size: int) -> tuple[torch.Tensor, torch.Tensor]:
            model.eval()
            all_maps = []
            all_labels = []
            with torch.no_grad():
                for inputs, labels in tqdm(loader, desc='maps', leave=False):
                    inputs = inputs.to(device, non_blocking=True)
                    anomaly_map = model.anomaly_map(inputs, output_size=image_size)
                    all_maps.append(anomaly_map.cpu())
                    all_labels.append(labels.cpu())
            return (torch.cat(all_maps, dim=0), torch.cat(all_labels, dim=0))

        def evaluate_score_variants(val_maps: torch.Tensor, test_maps: torch.Tensor, test_labels: torch.Tensor, scoring_config: dict[str, Any]) -> pd.DataFrame:
            labels_np = test_labels.numpy().astype(int)
            rows = []
            for variant in scoring_config['sweep_variants']:
                val_scores = reduce_anomaly_map(val_maps, variant['reduction'], variant['topk_ratio']).numpy()
                test_scores = reduce_anomaly_map(test_maps, variant['reduction'], variant['topk_ratio']).numpy()
                threshold = float(np.quantile(val_scores, scoring_config['threshold_quantile']))
                summary = summarize_threshold_metrics(labels_np, test_scores, threshold)
                _, best_sweep = sweep_threshold_metrics(labels_np, test_scores)
                rows.append({'name': variant['name'], 'reduction': variant['reduction'], 'topk_ratio': float(variant['topk_ratio']), 'threshold': threshold, 'precision': summary['precision'], 'recall': summary['recall'], 'f1': summary['f1'], 'auroc': summary['auroc'], 'auprc': summary['auprc'], 'predicted_anomalies': summary['predicted_anomalies'], 'best_sweep_threshold': best_sweep['threshold'], 'best_sweep_precision': best_sweep['precision'], 'best_sweep_recall': best_sweep['recall'], 'best_sweep_f1': best_sweep['f1']})
            return pd.DataFrame(rows).sort_values(['f1', 'auprc', 'auroc'], ascending=False).reset_index(drop=True)

        def train_fastflow(config: dict[str, Any]) -> dict[str, Any]:
            set_seed(config['run']['seed'])
            device = resolve_device(config['training']['device'])
            output_dir = REPO_ROOT / config['run']['output_dir']
            output_dir.mkdir(parents=True, exist_ok=True)
            print(f'Using device: {device}')
            print(f'Saving artifacts to: {output_dir}')
            loaders = make_dataloaders(config, device)
            backbone = FrozenWideResNet50Backbone(layers=config['model']['layers'], input_size=config['model']['input_size'], pretrained=config['model']['pretrained'])
            model = FastFlowModel(backbone=backbone, image_size=config['data']['image_size'], flow_steps=config['model']['flow_steps'], hidden_channels=config['model']['hidden_channels'], scale_clamp=config['model']['scale_clamp']).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=config['training']['learning_rate'], weight_decay=config['training']['weight_decay'])
            scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda' and config['training']['amp'])
            history: list[dict[str, Any]] = []
            best_state = None
            best_val_loss = float('inf')
            best_epoch = 0
            epochs_without_improvement = 0
            if config['run']['run_training']:
                for epoch in range(1, config['training']['epochs'] + 1):
                    started = time.time()
                    train_loss = run_epoch(model, loaders['train'], device, optimizer=optimizer, scaler=scaler, amp_enabled=device.type == 'cuda' and config['training']['amp'], grad_clip_norm=config['training']['grad_clip_norm'])
                    val_loss = run_epoch(model, loaders['val'], device, optimizer=None, scaler=scaler, amp_enabled=False, grad_clip_norm=None)
                    epoch_record = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'elapsed_seconds': time.time() - started}
                    history.append(epoch_record)
                    print(epoch_record)
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        best_epoch = epoch
                        best_state = copy.deepcopy(model.state_dict())
                        epochs_without_improvement = 0
                    else:
                        epochs_without_improvement += 1
                        if epochs_without_improvement >= config['training']['early_stopping_patience']:
                            print(f'Early stopping at epoch {epoch}')
                            break
                if best_state is not None:
                    model.load_state_dict(best_state)
            checkpoint_path = output_dir / 'best_model.pt'
            torch.save({'model_state_dict': model.state_dict(), 'config': config, 'best_epoch': best_epoch}, checkpoint_path)
            history_df = pd.DataFrame(history)
            history_path = output_dir / 'history.csv'
            history_df.to_csv(history_path, index=False)
            val_maps, _ = collect_anomaly_maps(model, loaders['val'], device=device, image_size=config['data']['image_size'])
            test_maps, test_labels = collect_anomaly_maps(model, loaders['test'], device=device, image_size=config['data']['image_size'])
            score_results = evaluate_score_variants(val_maps, test_maps, test_labels, config['scoring'])
            score_results_path = output_dir / 'score_variants.csv'
            score_results.to_csv(score_results_path, index=False)
            default_variant = score_results[(score_results['reduction'] == config['scoring']['default_reduction']) & np.isclose(score_results['topk_ratio'], config['scoring']['default_topk_ratio'])].iloc[0]
            best_variant = score_results.iloc[0]
            metrics = {'best_epoch': int(best_epoch), 'best_val_loss': float(best_val_loss), 'default_variant': default_variant.to_dict(), 'best_variant': best_variant.to_dict(), 'checkpoint_path': str(checkpoint_path.relative_to(REPO_ROOT)), 'history_path': str(history_path.relative_to(REPO_ROOT)), 'score_results_path': str(score_results_path.relative_to(REPO_ROOT))}
            metrics_path = output_dir / 'metrics.json'
            metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
            return {'model': model, 'device': device, 'history': history_df, 'val_maps': val_maps, 'test_maps': test_maps, 'test_labels': test_labels, 'score_results': score_results, 'metrics': metrics, 'output_dir': output_dir}
    else:
        print('[WARNING] RUN_TRAINING is False. Skipping this training section.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if run_training:\n    def make_dataloaders(config: dict[str, any], device: torch.device) -> dict[str, dataloader]:\n        metadata_csv = str(repo_root / config['data']['metadata_csv'])\n        image_size = config['data']['image_size']\n        batch_size = config['data']['batch_size']\n        num_workers = config['data']['num_workers']\n        pin_memory = device.type == 'cuda'\n        train_dataset = wafermapdataset(metadata_csv=metadata_csv, split='train', image_size=image_size)\n        val_dataset = wafermapdataset(metadata_csv=metadata_csv, split='val', image_size=image_size)\n        test_dataset = wafermapdataset(metadata_csv=metadata_csv, split='test', image_size=image_size)\n        return {'train': dataloader(train_dataset, batch_size=batch_size, shuffle=true, num_workers=num_workers, pin_memory=pin_memory), 'val': dataloader(val_dataset, batch_size=batch_size, shuffle=false, num_workers=num_workers, pin_memory=pin_memory), 'test': dataloader(test_dataset, batch_size=batch_size, shuffle=false, num_workers=num_workers, pin_memory=pin_memory)}\n\n    def run_epoch(model: fastflowmodel, loader: dataloader, device: torch.device, optimizer: torch.optim.optimizer | none, scaler: torch.cuda.amp.gradscaler, amp_enabled: bool, grad_clip_norm: float | none) -> float:\n        is_training = optimizer is not none\n        model.train(is_training)\n        running_loss = 0.0\n        total_items = 0\n        for inputs, _ in tqdm(loader, desc='train' if is_training else 'eval', leave=false):\n            inputs = inputs.to(device, non_blocking=true)\n            with torch.set_grad_enabled(is_training):\n                with torch.cuda.amp.autocast(enabled=amp_enabled):\n                    zs, logdets = model(inputs)\n                    loss = compute_fastflow_loss(zs, logdets)\n                if is_training:\n                    optimizer.zero_grad(set_to_none=true)\n                    scaler.scale(loss).backward()\n                    if grad_clip_norm is not none:\n                        scaler.unscale_(optimizer)\n                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)\n                    scaler.step(optimizer)\n                    scaler.update()\n            batch_size = inputs.shape[0]\n            running_loss += float(loss.detach().cpu()) * batch_size\n            total_items += batch_size\n        return running_loss / max(1, total_items)\n\n    def collect_anomaly_maps(model: fastflowmodel, loader: dataloader, device: torch.device, image_size: int) -> tuple[torch.tensor, torch.tensor]:\n        model.eval()\n        all_maps = []\n        all_labels = []\n        with torch.no_grad():\n            for inputs, labels in tqdm(loader, desc='maps', leave=false):\n                inputs = inputs.to(device, non_blocking=true)\n                anomaly_map = model.anomaly_map(inputs, output_size=image_size)\n                all_maps.append(anomaly_map.cpu())\n                all_labels.append(labels.cpu())\n        return (torch.cat(all_maps, dim=0), torch.cat(all_labels, dim=0))\n\n    def evaluate_score_variants(val_maps: torch.tensor, test_maps: torch.tensor, test_labels: torch.tensor, scoring_config: dict[str, any]) -> pd.dataframe:\n        labels_np = test_labels.numpy().astype(int)\n        rows = []\n        for variant in scoring_config['sweep_variants']:\n            val_scores = reduce_anomaly_map(val_maps, variant['reduction'], variant['topk_ratio']).numpy()\n            test_scores = reduce_anomaly_map(test_maps, variant['reduction'], variant['topk_ratio']).numpy()\n            threshold = float(np.quantile(val_scores, scoring_config['threshold_quantile']))\n            summary = summarize_threshold_metrics(labels_np, test_scores, threshold)\n            _, best_sweep = sweep_threshold_metrics(labels_np, test_scores)\n            rows.append({'name': variant['name'], 'reduction': variant['reduction'], 'topk_ratio': float(variant['topk_ratio']), 'threshold': threshold, 'precision': summary['precision'], 'recall': summary['re"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
results = train_fastflow(CONFIG)
display(results['history'].tail())
display(results['score_results'])
results['metrics']

In [ ]:
try:
    history = results['history']
    score_results = results['score_results']
    test_maps = results['test_maps']
    test_labels = results['test_labels'].numpy()
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    if not history.empty:
        history.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0], marker='o')
        axes[0].set_title('FastFlow Loss by Epoch')
        axes[0].grid(alpha=0.3)
    else:
        axes[0].set_title('No training history recorded')
        axes[0].axis('off')
    score_results.plot.bar(x='name', y='f1', ax=axes[1], color='#1f77b4')
    axes[1].set_title('Validation-Threshold Test F1 by Score Variant')
    axes[1].set_ylabel('F1')
    axes[1].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    example_indices = list(np.where(test_labels == 0)[0][:3]) + list(np.where(test_labels == 1)[0][:3])
    fig, axes = plt.subplots(len(example_indices), 2, figsize=(8, 3 * len(example_indices)))
    for row_index, example_index in enumerate(example_indices):
        anomaly_map = test_maps[example_index, 0].numpy()
        label_name = 'anomaly' if test_labels[example_index] == 1 else 'normal'
        axes[row_index, 0].imshow(anomaly_map, cmap='magma')
        axes[row_index, 0].set_title(f'Anomaly map: {label_name}')
        axes[row_index, 1].hist(anomaly_map.ravel(), bins=30, color='#ff7f0e')
        axes[row_index, 1].set_title('Anomaly-map value histogram')
        axes[row_index, 0].set_xticks([])
        axes[row_index, 0].set_yticks([])
    plt.tight_layout()
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "history = results['history']\nscore_results = results['score_results']\ntest_maps = results['test_maps']\ntest_labels = results['test_labels'].numpy()\nfig, axes = plt.subplots(1, 2, figsize=(14, 4))\nif not history.empty:\n    history.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0], marker='o')\n    axes[0].set_title('fastflow loss by epoch')\n    axes[0].grid(alpha=0.3)\nelse:\n    axes[0].set_title('no training history recorded')\n    axes[0].axis('off')\nscore_results.plot.bar(x='name', y='f1', ax=axes[1], color='#1f77b4')\naxes[1].set_title('validation-threshold test f1 by score variant')\naxes[1].set_ylabel('f1')\naxes[1].grid(axis='y', alpha=0.3)\nplt.tight_layout()\nplt.show()\nexample_indices = list(np.where(test_labels == 0)[0][:3]) + list(np.where(test_labels == 1)[0][:3])\nfig, axes = plt.subplots(len(example_indices), 2, figsize=(8, 3 * len(example_indices)))\nfor row_index, example_index in enumerate(example_indices):\n    anomaly_map = test_maps[example_index, 0].numpy()\n    label_name = 'anomaly' if test_labels[example_index] == 1 else 'normal'\n    axes[row_index, 0].imshow(anomaly_map, cmap='magma')\n    axes[row_index, 0].set_title(f'anomaly map: {label_name}')\n    axes[row_index, 1].hist(anomaly_map.ravel(), bins=30, color='#ff7f0e')\n    axes[row_index, 1].set_title('anomaly-map value histogram')\n    axes[row_index, 0].set_xticks([])\n    axes[row_index, 0].set_yticks([])\nplt.tight_layout()\nplt.show()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Notes

- The default comparison variant in this notebook is `topk_mean` with ratio `0.10` because your best PatchCore results also won with a local top-k reduction.
- The exported `score_variants.csv` is there on purpose: score reduction can change the ranking a lot in this repo, so do not judge FastFlow from only one reduction rule.
- If you want a stronger FastFlow follow-up later, the first things worth varying are the selected feature layers, number of flow steps, and whether you score each layer separately before fusion.